# Modelos Baseline para Detección de Anomalías

Notebook de comparación de modelos baseline contra el modelo DNF (Deep Neuro-Fuzzy) para establecer referencias de rendimiento.

**Contenido del notebook:**
- **Baseline 1: Random Forest** — Clasificador basado en árboles de decisión ensemble como referencia clásica de machine learning.
- **Baseline 2: LSTM** — Red neuronal recurrente simple para capturar patrones temporales en las secuencias de sensores.
- **Baseline 3: Gradient Boosting (XGBoost)** — Clasificador ensemble basado en boosting como referencia de alto rendimiento.
- **Baseline 4: ANFIS** — Adaptive Neuro-Fuzzy Inference System como referencia del enfoque neuro-difuso sin la rama deep learning.


In [1]:
import numpy as np
import pandas as pd
import json
import os
import yaml
import logging
import pickle
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report, roc_auc_score
)
from xgboost import XGBClassifier
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)


def load_config(config_path: str = "config/params.yaml") -> dict:
    with open(config_path, "r") as f:
        return yaml.safe_load(f)


def get_project_paths(config_path: str) -> dict:
    """Resolve all project directories from a config file path."""
    project_dir = os.path.abspath(os.path.join(os.path.dirname(config_path), ".."))
    return {
        "project_dir":   project_dir,
        "processed_dir": os.path.join(project_dir, "data", "processed"),
        "splits_dir":    os.path.join(project_dir, "data", "splits"),
        "model_dir":     os.path.join(project_dir, "models", "artifacts", "baseline_models"),
    }


def load_csv_splits(splits_dir: str):
    """Load train/val/test CSV splits from the splits directory."""
    logger.info("Loading CSV splits...")
    train_df = pd.read_csv(os.path.join(splits_dir, "train.csv"))
    val_df   = pd.read_csv(os.path.join(splits_dir, "val.csv"))
    test_df  = pd.read_csv(os.path.join(splits_dir, "test.csv"))
    return train_df, val_df, test_df


def extract_features_csv(train_df, val_df, test_df, target_col="fault_name"):
    """Extract feature matrices and binary labels from CSV DataFrames."""
    drop_cols = {target_col, "timestamp", "cycle_id"}
    feature_cols = [c for c in train_df.columns if c not in drop_cols]

    X_train = train_df[feature_cols].to_numpy(dtype=np.float32)
    X_val   = val_df[feature_cols].to_numpy(dtype=np.float32)
    X_test  = test_df[feature_cols].to_numpy(dtype=np.float32)

    y_train = (train_df[target_col].astype(str).str.strip().str.upper() != "NORMAL").astype(int).to_numpy()
    y_val   = (val_df[target_col].astype(str).str.strip().str.upper() != "NORMAL").astype(int).to_numpy()
    y_test  = (test_df[target_col].astype(str).str.strip().str.upper() != "NORMAL").astype(int).to_numpy()

    return X_train, X_val, X_test, y_train, y_val, y_test, feature_cols


def compute_metrics(y_true, y_pred, y_score=None):
    """Compute binary classification metrics (aligned with get_stats.py)."""
    from sklearn.metrics import confusion_matrix as sk_confusion_matrix
    tn, fp, fn, tp = sk_confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    prec_fallo = float(tp / max(tp + fp, 1))
    rec_fallo = float(tp / max(tp + fn, 1))
    prec_nofallo = float(tn / max(tn + fn, 1))
    rec_nofallo = float(tn / max(tn + fp, 1))
    f1_fallo = float(f1_score(y_true, y_pred, zero_division=0))
    f1_nofallo = float(2 * prec_nofallo * rec_nofallo / max(prec_nofallo + rec_nofallo, 1e-9))
    macro_precision = float((prec_nofallo + prec_fallo) / 2)
    macro_recall = float((rec_nofallo + rec_fallo) / 2)
    macro_f1 = float((f1_nofallo + f1_fallo) / 2)
    nofallo_specificity = float(tn / max(tn + fp, 1))
    auc_roc = None
    if y_score is not None and len(np.unique(y_true)) == 2:
        try:
            auc_roc = float(roc_auc_score(y_true, np.asarray(y_score).reshape(-1)))
        except ValueError:
            auc_roc = None
    return {
        "accuracy":                float(accuracy_score(y_true, y_pred)),
        "fallo_f1":                f1_fallo,
        "fallo_precision":         prec_fallo,
        "fallo_recall":            rec_fallo,
        "nofallo_f1":              f1_nofallo,
        "nofallo_precision":       prec_nofallo,
        "nofallo_recall":          rec_nofallo,
        "nofallo_specificity":     nofallo_specificity,
        "auc_roc":                 auc_roc,
        "macro_f1":                macro_f1,
        "macro_precision":         macro_precision,
        "macro_recall":            macro_recall,
        "tp": int(tp),
        "fp": int(fp),
        "tn": int(tn),
        "fn": int(fn),
    }


def evaluate_model(model, X_train, y_train, X_val, y_val, X_test, y_test, model_name="Model"):
    """Evaluate a model on train/val/test and return results dict."""
    results = {}
    for name, X, y in [("train", X_train, y_train),
                       ("val", X_val, y_val),
                       ("test", X_test, y_test)]:
        preds = model.predict(X).astype(int)
        y_score = None
        if hasattr(model, "predict_proba"):
            proba = model.predict_proba(X)
            y_score = proba[:, 1] if proba.ndim == 2 and proba.shape[1] > 1 else proba.ravel()
        elif hasattr(model, "decision_function"):
            y_score = model.decision_function(X)
        metrics = compute_metrics(y, preds, y_score)
        results[name] = metrics
        logger.info(f"{model_name} {name}: Acc={metrics['accuracy']:.4f}, F1_fallo={metrics['fallo_f1']:.4f}, F1_macro={metrics['macro_f1']:.4f}")
    return results

# Baseline 1: Random Forest

In [2]:
def train_random_forest(config_path: str = "../../config/config.yaml"):
    """Train RF baseline using CSV splits (binary target)."""
    config = load_config(config_path)
    paths = get_project_paths(config_path)
    splits_dir, model_dir = paths["splits_dir"], paths["model_dir"]
    os.makedirs(model_dir, exist_ok=True)

    train_df, val_df, test_df = load_csv_splits(splits_dir)
    X_train, X_val, X_test, y_train, y_val, y_test, feature_cols = extract_features_csv(
        train_df, val_df, test_df
    )
    logger.info(f"Feature vector size: {X_train.shape[1]}")

    logger.info("Training Random Forest...")
    rf = RandomForestClassifier(
        n_estimators=100,
        max_depth=7,
        min_samples_split=3,
        min_samples_leaf=2,
        random_state=config["project"]["seed"],
        n_jobs=-1,
    )
    rf.fit(X_train, y_train)

    results = evaluate_model(rf, X_train, y_train, X_val, y_val, X_test, y_test, "RF")

    rf_path = os.path.join(model_dir, "baseline_rf.pkl")
    with open(rf_path, "wb") as f:
        pickle.dump(rf, f)

    metrics_path = os.path.join(model_dir, "baseline_rf_metrics.json")
    with open(metrics_path, "w") as f:
        json.dump(results, f, indent=2)

    logger.info(f"Random Forest baseline saved to {rf_path}")
    return results

In [3]:
results = train_random_forest("../config/config.yaml")
results

2026-07-08 10:29:35,517 - INFO - Loading CSV splits...


2026-07-08 10:29:40,618 - INFO - Feature vector size: 13
2026-07-08 10:29:40,619 - INFO - Training Random Forest...
2026-07-08 10:30:29,770 - INFO - RF train: Acc=0.9178, F1_fallo=0.4994, F1_macro=0.7273
2026-07-08 10:30:30,196 - INFO - RF val: Acc=0.9636, F1_fallo=0.5153, F1_macro=0.7482
2026-07-08 10:30:30,823 - INFO - RF test: Acc=0.9612, F1_fallo=0.4201, F1_macro=0.7000
2026-07-08 10:30:30,838 - INFO - Random Forest baseline saved to c:\Users\julia\Documents\Proyectos-Hector\REAL DATAGIA\DataGIA\models\artifacts\baseline_models\baseline_rf.pkl


{'train': {'accuracy': 0.9177725,
  'fallo_f1': 0.4993784912303844,
  'fallo_precision': 0.977632101708383,
  'fallo_recall': 0.33533431906296635,
  'nofallo_f1': 0.9552076604510692,
  'nofallo_precision': 0.9151514360767531,
  'nofallo_recall': 0.9989309169143945,
  'nofallo_specificity': 0.9989309169143945,
  'auc_roc': 0.8593232824456609,
  'macro_f1': 0.7272930758407268,
  'macro_precision': 0.946391768892568,
  'macro_recall': 0.6671326179886804,
  'tp': 49214,
  'fp': 1126,
  'tn': 1052113,
  'fn': 97547},
 'val': {'accuracy': 0.9635916666666666,
  'fallo_f1': 0.5152557417064241,
  'fallo_precision': 0.9439024390243902,
  'fallo_recall': 0.35434152296658017,
  'nofallo_f1': 0.9810855156350791,
  'nofallo_precision': 0.9640037434065,
  'nofallo_recall': 0.9987835729459571,
  'nofallo_specificity': 0.9987835729459571,
  'auc_roc': 0.8647828414855133,
  'macro_f1': 0.7481706286707516,
  'macro_precision': 0.953953091215445,
  'macro_recall': 0.6765625479562687,
  'tp': 4644,
  'fp':

# Baseline 2: LSTM

In [4]:
class SimpleLSTM(nn.Module):
    """Simple LSTM binary classifier (0=NORMAL, 1=FAULT)."""

    def __init__(self, input_features: int = 13, hidden_size: int = 32, num_layers: int = 1):
        super().__init__()
        self.lstm = nn.LSTM(input_features, hidden_size, num_layers, batch_first=True)
        self.classifier = nn.Linear(32, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        lstm_out, _ = self.lstm(x)
        last = lstm_out[:, -1, :]
        return self.classifier(last).squeeze(1)  # shape: (batch,)


def train_simple_lstm(config_path: str = "../../config/config.yaml"):
    """Train binary LSTM baseline on sequence splits (0=NORMAL, 1=FAULT)."""
    config = load_config(config_path)
    paths = get_project_paths(config_path)
    processed_dir, model_dir = paths["processed_dir"], paths["model_dir"]
    os.makedirs(model_dir, exist_ok=True)
    seed = config["project"]["seed"]

    torch.manual_seed(seed)
    np.random.seed(seed)
    device = torch.device(
        "cuda" if torch.cuda.is_available()
        else ("mps" if torch.backends.mps.is_available() else "cpu")
    )
    logger.info(f"Device: {device}")

    # Load sequence data — y arrays are already binary (0=NORMAL, 1=FAULT)
    X_train = np.load(os.path.join(processed_dir, "X_train.npy"))
    y_train = np.load(os.path.join(processed_dir, "y_train.npy")).reshape(-1).astype(np.float32)
    X_val = np.load(os.path.join(processed_dir, "X_val.npy"))
    y_val = np.load(os.path.join(processed_dir, "y_val.npy")).reshape(-1).astype(np.float32)
    X_test = np.load(os.path.join(processed_dir, "X_test.npy"))
    y_test = np.load(os.path.join(processed_dir, "y_test.npy")).reshape(-1).astype(np.float32)

    unique_labels = np.unique(np.concatenate([y_train, y_val, y_test]))
    logger.info(f"Binary LSTM — unique labels in splits: {unique_labels}")
    assert set(unique_labels).issubset({0.0, 1.0}), (", "
        f"Expected binary labels (0/1), found {unique_labels}"
    )

    input_features = X_train.shape[2]
    logger.info(f"Sequences: {X_train.shape}, input_features={input_features}")

    train_ds = TensorDataset(torch.FloatTensor(X_train), torch.FloatTensor(y_train))
    val_ds   = TensorDataset(torch.FloatTensor(X_val),   torch.FloatTensor(y_val))
    test_ds  = TensorDataset(torch.FloatTensor(X_test),  torch.FloatTensor(y_test))

    train_loader = DataLoader(train_ds, batch_size=64, shuffle=False, num_workers=0, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=64, shuffle=False, num_workers=0, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=64, shuffle=False, num_workers=0, pin_memory=True)

    model = SimpleLSTM(
        input_features=input_features,
        hidden_size=32,   
        num_layers=1,
    ).to(device)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001) 

    best_val_f1 = 0.0
    lstm_ckpt = os.path.join(model_dir, "baseline_lstm.pt")
    history = []
    for epoch in range(1, 20):   
        model.train()
        for X_b, y_b in train_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            logits = model(X_b)
            loss = criterion(logits, y_b)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for X_b, y_b in val_loader:
                logits = model(X_b.to(device))
                scores = logits.sigmoid().cpu().numpy()
                preds = (scores >= 0.5).astype(int)
                all_preds.extend(preds)
                all_labels.extend(y_b.long().numpy())

        val_f1  = f1_score(all_labels, all_preds, zero_division=0)
        val_acc = accuracy_score(all_labels, all_preds)

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            torch.save(model.state_dict(), lstm_ckpt)

        logger.info(f"LSTM Epoch {epoch}: Val Acc={val_acc:.4f}, Val F1_binary={val_f1:.4f}")
        history.append({"epoch": epoch, "val_acc": float(val_acc), "val_f1_binary": float(val_f1)})

    model.load_state_dict(torch.load(lstm_ckpt, weights_only=True))
    model.eval()

    results = {}
    for name, loader in [("train", train_loader), ("val", val_loader), ("test", test_loader)]:
        all_preds, all_labels, all_scores = [], [], []
        with torch.no_grad():
            for X_b, y_b in loader:
                logits = model(X_b.to(device))
                scores = logits.sigmoid().cpu().numpy()
                preds = (scores >= 0.5).astype(int)
                all_preds.extend(preds)
                all_labels.extend(y_b.long().numpy())
                all_scores.extend(scores)

        metrics = compute_metrics(np.array(all_labels), np.array(all_preds), np.array(all_scores))
        results[name] = metrics
        logger.info(f"LSTM {name}: Acc={metrics['accuracy']:.4f}, F1_fallo={metrics['fallo_f1']:.4f}, F1_macro={metrics['macro_f1']:.4f}")

    with open(os.path.join(model_dir, "baseline_lstm_metrics.json"), "w") as f:
        json.dump(results, f, indent=2)

    with open(os.path.join(model_dir, "baseline_lstm_history.json"), "w") as f:
        json.dump(history, f, indent=2)

    logger.info("Binary LSTM baseline completed.")
    return results


In [5]:
results = train_simple_lstm("../config/config.yaml")
results

2026-07-08 10:30:30,951 - INFO - Device: cuda


2026-07-08 10:30:31,157 - INFO - Binary LSTM — unique labels in splits: [0. 1.]
2026-07-08 10:30:31,158 - INFO - Sequences: (10000, 180, 13), input_features=13
2026-07-08 10:30:34,382 - INFO - LSTM Epoch 1: Val Acc=0.9405, Val F1_binary=0.1314
2026-07-08 10:30:35,049 - INFO - LSTM Epoch 2: Val Acc=0.9490, Val F1_binary=0.4270
2026-07-08 10:30:35,646 - INFO - LSTM Epoch 3: Val Acc=0.9520, Val F1_binary=0.4545
2026-07-08 10:30:36,321 - INFO - LSTM Epoch 4: Val Acc=0.9560, Val F1_binary=0.5600
2026-07-08 10:30:37,010 - INFO - LSTM Epoch 5: Val Acc=0.9560, Val F1_binary=0.5464
2026-07-08 10:30:37,707 - INFO - LSTM Epoch 6: Val Acc=0.9580, Val F1_binary=0.5882
2026-07-08 10:30:38,391 - INFO - LSTM Epoch 7: Val Acc=0.9525, Val F1_binary=0.4920
2026-07-08 10:30:39,044 - INFO - LSTM Epoch 8: Val Acc=0.9485, Val F1_binary=0.3522
2026-07-08 10:30:39,678 - INFO - LSTM Epoch 9: Val Acc=0.9500, Val F1_binary=0.4318
2026-07-08 10:30:40,420 - INFO - LSTM Epoch 10: Val Acc=0.9480, Val F1_binary=0.4023

{'train': {'accuracy': 0.9428,
  'fallo_f1': 0.7841509433962264,
  'fallo_precision': 0.7998460354118553,
  'fallo_recall': 0.769059955588453,
  'nofallo_f1': 0.9670317002881845,
  'nofallo_precision': 0.9641420526376279,
  'nofallo_recall': 0.9699387212394497,
  'nofallo_specificity': 0.9699387212394497,
  'auc_roc': 0.9352450992096655,
  'macro_f1': 0.8755913218422055,
  'macro_precision': 0.8819940440247416,
  'macro_recall': 0.8694993384139513,
  'tp': 1039,
  'fp': 260,
  'tn': 8389,
  'fn': 312},
 'val': {'accuracy': 0.9695,
  'fallo_f1': 0.7426160337552743,
  'fallo_precision': 0.7586206896551724,
  'fallo_recall': 0.7272727272727273,
  'nofallo_f1': 0.9837895296306138,
  'nofallo_precision': 0.982484076433121,
  'nofallo_recall': 0.9850984566258648,
  'nofallo_specificity': 0.9850984566258648,
  'auc_roc': 0.9487198659388896,
  'macro_f1': 0.8632027816929441,
  'macro_precision': 0.8705523830441466,
  'macro_recall': 0.856185591949296,
  'tp': 88,
  'fp': 28,
  'tn': 1851,
  'f

# Baseline 3: XGBoost

In [6]:
def train_xgboost(config_path: str = "../../config/config.yaml"):
    """Train XGBoost baseline using CSV splits (binary target)."""
    config = load_config(config_path)
    paths = get_project_paths(config_path)
    splits_dir, model_dir = paths["splits_dir"], paths["model_dir"]
    os.makedirs(model_dir, exist_ok=True)

    train_df, val_df, test_df = load_csv_splits(splits_dir)
    X_train, X_val, X_test, y_train, y_val, y_test, feature_cols = extract_features_csv(
        train_df, val_df, test_df
    )

    logger.info("Training XGBoost...")
    xgb = XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.01,
        random_state=config["project"]["seed"],
        tree_method="hist",
        device="cuda",
    )
    xgb.fit(X_train, y_train)

    results = evaluate_model(xgb, X_train, y_train, X_val, y_val, X_test, y_test, "XGB")

    with open(os.path.join(model_dir, "baseline_xgb.pkl"), "wb") as f:
        pickle.dump(xgb, f)
    with open(os.path.join(model_dir, "baseline_xgb_metrics.json"), "w") as f:
        json.dump(results, f, indent=2)

    return results

In [7]:
results = train_xgboost("../config/config.yaml")
results

2026-07-08 10:30:46,996 - INFO - Loading CSV splits...


2026-07-08 10:30:51,983 - INFO - Training XGBoost...
c:\Users\julia\Documents\Proyectos-Hector\REAL DATAGIA\DataGIA\env\Lib\site-packages\xgboost\core.py:751: UserWarning: [10:30:53] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)
2026-07-08 10:30:54,738 - INFO - XGB train: Acc=0.9215, F1_fallo=0.5297, F1_macro=0.7434
2026-07-08 10:30:54,939 - INFO - XGB val: Acc=0.9656, F1_fallo=0.5484, F1_macro=0.7653
2026-07-08 10:30:55,242 - INFO - XGB test: Acc=0.9645, F1_fallo=0.4861, F1_macro=0.7339


{'train': {'accuracy': 0.9214575,
  'fallo_f1': 0.5296504229358485,
  'fallo_precision': 0.9896128599134716,
  'fallo_recall': 0.361587887790353,
  'nofallo_f1': 0.9571511378127536,
  'nofallo_precision': 0.9182693985219509,
  'nofallo_recall': 0.9994711551699093,
  'nofallo_specificity': 0.9994711551699093,
  'auc_roc': 0.8676105007458308,
  'macro_f1': 0.743400780374301,
  'macro_precision': 0.9539411292177112,
  'macro_recall': 0.6805295214801311,
  'tp': 53067,
  'fp': 557,
  'tn': 1052682,
  'fn': 93694},
 'val': {'accuracy': 0.9656416666666666,
  'fallo_f1': 0.5483623617044583,
  'fallo_precision': 0.9716614906832298,
  'fallo_recall': 0.3819624599420113,
  'nofallo_f1': 0.9821415422465359,
  'nofallo_precision': 0.9655096062133806,
  'nofallo_recall': 0.9993565277177889,
  'nofallo_specificity': 0.9993565277177889,
  'auc_roc': 0.8744147395701808,
  'macro_f1': 0.7652519519754971,
  'macro_precision': 0.9685855484483052,
  'macro_recall': 0.6906594938299001,
  'tp': 5006,
  'fp'

# Baseline 4: ANFIS

In [8]:
if 'logger' not in globals():
    logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
    logger = logging.getLogger(__name__)

# ===============================
# ANFIS baseline (binary)
# ===============================

class GaussianMembershipFunction(nn.Module):
    def __init__(self, n_inputs, n_mf_per_input):
        super().__init__()
        centers = torch.linspace(-2.5, 2.5, n_mf_per_input).repeat(n_inputs, 1)
        self.centers = nn.Parameter(centers)
        self.sigmas = nn.Parameter(torch.ones(n_inputs, n_mf_per_input) * 0.9)

    def forward(self, x):
        x_expanded = x.unsqueeze(2)
        sig = torch.clamp(torch.abs(self.sigmas), min=1e-4)
        return torch.exp(-((x_expanded - self.centers) ** 2) / (2 * sig ** 2 + 1e-8))


class FuzzyRuleLayer(nn.Module):
    def __init__(self, n_inputs, n_mf_per_input, n_rules):
        super().__init__()
        self.n_inputs = n_inputs
        self.rule_weights = nn.Parameter(
            torch.randn(n_rules, n_inputs, n_mf_per_input) * 0.06
        )

    def forward(self, memberships):
        attn = F.softmax(self.rule_weights, dim=-1)
        m = memberships.unsqueeze(1)
        selected = (m * attn).sum(dim=-1)  # [B,R,F]
        firing = torch.exp(torch.log(selected.clamp_min(1e-8)).mean(dim=-1))
        return firing


class ANFIS(nn.Module):
    def __init__(self, n_inputs, n_mf_per_input=3, n_rules=40):
        super().__init__()
        self.membership_layer = GaussianMembershipFunction(n_inputs, n_mf_per_input)
        self.rule_layer = FuzzyRuleLayer(n_inputs, n_mf_per_input, n_rules)
        self.consequent_network = nn.Sequential(
            nn.Linear(n_rules, 128), nn.ReLU(), nn.BatchNorm1d(128), nn.Dropout(0.15),
            nn.Linear(128, 64), nn.ReLU(), nn.BatchNorm1d(64), nn.Dropout(0.10),
        )
        self.binary_head = nn.Linear(64, 1)

    def forward(self, x):
        m = self.membership_layer(x)
        r = self.rule_layer(m)
        r = r / (r.sum(dim=1, keepdim=True) + 1e-8)
        h = self.consequent_network(r)
        return self.binary_head(h).squeeze(-1)  # shape: (batch,)


def _build_interpretable_temporal_features(X_seq: np.ndarray) -> np.ndarray:
    """Features por sensor (interpretables): mean, std, rango, delta final-inicial."""
    mean = X_seq.mean(axis=1)
    std = X_seq.std(axis=1)
    value_range = X_seq.max(axis=1) - X_seq.min(axis=1)
    delta = X_seq[:, -1, :] - X_seq[:, 0, :]
    return np.concatenate([mean, std, value_range, delta], axis=1).astype(np.float32)


def run_anfis_pipeline(
    processed_dir: str,
    model_dir: str,
    seed: int = 42,
    n_epochs: int = 90,
    patience: int = 18,
    lr: float = 4e-4,
    weight_decay: float = 1e-4,
    n_mf_per_input: int = 3,
    n_rules: int = 40,
):
    """Pipeline ANFIS: clasificación binaria (0=NORMAL, 1=FAULT)."""
    os.makedirs(model_dir, exist_ok=True)

    torch.manual_seed(seed)
    np.random.seed(seed)

    device = torch.device(
        "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
    )
    logger.info(f"ANFIS Device: {device}")

    X_train = np.load(os.path.join(processed_dir, "X_train.npy"), allow_pickle=True)
    y_train = np.load(os.path.join(processed_dir, "y_train.npy"), allow_pickle=True)
    X_val = np.load(os.path.join(processed_dir, "X_val.npy"), allow_pickle=True)
    y_val = np.load(os.path.join(processed_dir, "y_val.npy"), allow_pickle=True)
    X_test = np.load(os.path.join(processed_dir, "X_test.npy"), allow_pickle=True)
    y_test = np.load(os.path.join(processed_dir, "y_test.npy"), allow_pickle=True)

    if X_train.dtype == object:
        X_train = X_train.astype(np.float32)
    if X_val.dtype == object:
        X_val = X_val.astype(np.float32)
    if X_test.dtype == object:
        X_test = X_test.astype(np.float32)

    Xtr = _build_interpretable_temporal_features(X_train)
    Xva = _build_interpretable_temporal_features(X_val)
    Xte = _build_interpretable_temporal_features(X_test)

    mu = Xtr.mean(axis=0, keepdims=True)
    sigma = Xtr.std(axis=0, keepdims=True) + 1e-6
    Xtr = (Xtr - mu) / sigma
    Xva = (Xva - mu) / sigma
    Xte = (Xte - mu) / sigma

    # Binary labels (0=NORMAL, 1=FAULT)
    y_train = (y_train > 0).astype(np.float32)
    y_val = (y_val > 0).astype(np.float32)
    y_test = (y_test > 0).astype(np.float32)

    pos = float(y_train.sum())
    neg = float(len(y_train) - pos)
    pos_weight_value = (neg / max(pos, 1.0)) if pos > 0 else 1.0
    pos_weight = torch.tensor([pos_weight_value], dtype=torch.float32, device=device)

    train_ds = TensorDataset(torch.tensor(Xtr, dtype=torch.float32), torch.tensor(y_train, dtype=torch.float32))
    val_ds = TensorDataset(torch.tensor(Xva, dtype=torch.float32), torch.tensor(y_val, dtype=torch.float32))
    test_ds = TensorDataset(torch.tensor(Xte, dtype=torch.float32), torch.tensor(y_test, dtype=torch.float32))

    batch_size = 256 if device.type == "cuda" else 128
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=(device.type == "cuda"))
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=(device.type == "cuda"))
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=(device.type == "cuda"))

    model = ANFIS(n_inputs=Xtr.shape[1], n_mf_per_input=n_mf_per_input, n_rules=n_rules).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=4)

    def run_epoch(loader, train=False):
        model.train() if train else model.eval()
        total_loss = 0.0
        all_preds, all_labels, all_scores = [], [], []

        with torch.set_grad_enabled(train):
            for xb, yb in loader:
                xb = xb.to(device, non_blocking=True)
                yb = yb.to(device, non_blocking=True)

                if train:
                    optimizer.zero_grad(set_to_none=True)

                logits = model(xb)
                loss = criterion(logits, yb)

                if train:
                    loss.backward()
                    nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    optimizer.step()

                total_loss += loss.item()
                scores = torch.sigmoid(logits).detach().cpu().numpy()
                preds = (scores >= 0.5).astype(int)
                all_preds.extend(preds)
                all_labels.extend(yb.long().cpu().numpy())
                all_scores.extend(scores)

        avg_loss = total_loss / max(len(loader), 1)
        acc = accuracy_score(all_labels, all_preds)
        f1m = f1_score(all_labels, all_preds, average="macro", zero_division=0)
        return {
            "loss": float(avg_loss),
            "acc": float(acc),
            "f1_macro": float(f1m),
            "preds": np.array(all_preds),
            "labels": np.array(all_labels),
            "scores": np.array(all_scores),
        }

    history = []
    best_f1 = -1.0
    best_state = None
    best_epoch = 0
    no_improve = 0

    for epoch in range(1, n_epochs + 1):
        tr = run_epoch(train_loader, train=True)
        va = run_epoch(val_loader, train=False)
        scheduler.step(va["f1_macro"])

        history.append({
            "epoch": epoch,
            "train_loss": tr["loss"],
            "train_acc": tr["acc"],
            "train_f1_macro": tr["f1_macro"],
            "val_loss": va["loss"],
            "val_acc": va["acc"],
            "val_f1_macro": va["f1_macro"],
        })

        logger.info(
            f"ANFIS Epoch {epoch:02d}: tr_loss={tr['loss']:.4f}, tr_f1={tr['f1_macro']:.4f}, "
            f"va_loss={va['loss']:.4f}, va_f1={va['f1_macro']:.4f}"
        )

        if va["f1_macro"] > best_f1:
            best_f1 = va["f1_macro"]
            best_epoch = epoch
            no_improve = 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            no_improve += 1
            if no_improve >= patience:
                logger.info("ANFIS early stopping.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    metrics = {}
    for name, loader in [("train", train_loader), ("val", val_loader), ("test", test_loader)]:
        out = run_epoch(loader, train=False)
        labels, preds, scores = out["labels"], out["preds"], out["scores"]

        cm = confusion_matrix(labels, preds, labels=[0, 1])
        class_report = classification_report(labels, preds, output_dict=True, zero_division=0)

        cm_vals = cm.ravel()
        tn, fp, fn, tp = cm_vals
        base = compute_metrics(labels, preds, scores)
        metrics[name] = {
            **base,
            "loss": out["loss"],
            "confusion_matrix": cm.tolist(),
            "classification_report": class_report,
        }
        logger.info(
            f"ANFIS {name}: Acc={metrics[name]['accuracy']:.4f}, F1_fallo={metrics[name]['fallo_f1']:.4f}, F1_macro={metrics[name]['macro_f1']:.4f}"
        )

    ckpt = {
        "model_state_dict": model.state_dict(),
        "n_inputs": int(Xtr.shape[1]),
        "mu": mu.astype(np.float32),
        "sigma": sigma.astype(np.float32),
        "best_epoch": int(best_epoch),
        "best_val_f1": float(best_f1),
    }
    torch.save(ckpt, os.path.join(model_dir, "baseline_anfis.pt"))

    with open(os.path.join(model_dir, "baseline_anfis_metrics.json"), "w") as f:
        json.dump(metrics, f, indent=2)

    with open(os.path.join(model_dir, "baseline_anfis_history.json"), "w") as f:
        json.dump(history, f, indent=2)

    return {
        "best_epoch": int(best_epoch),
        "best_val_f1": float(best_f1),
        "history": history,
        "metrics": metrics,
    }


def train_anfis_baseline(config_path: str = "config/params.yaml"):
    """Wrapper de compatibilidad: mantiene tu llamada actual, pero usa pipeline estilo final."""
    config = load_config(config_path)
    paths = get_project_paths(config_path)
    return run_anfis_pipeline(
        processed_dir=paths["processed_dir"],
        model_dir=paths["model_dir"],
        seed=config["project"]["seed"],
    )

In [9]:
results = train_anfis_baseline("../config/config.yaml")
results

2026-07-08 10:30:55,357 - INFO - ANFIS Device: cuda


2026-07-08 10:30:56,512 - INFO - ANFIS Epoch 01: tr_loss=1.0422, tr_f1=0.4321, va_loss=0.9188, va_f1=0.7491
2026-07-08 10:30:56,900 - INFO - ANFIS Epoch 02: tr_loss=0.9374, tr_f1=0.6080, va_loss=0.7747, va_f1=0.4844
2026-07-08 10:30:57,393 - INFO - ANFIS Epoch 03: tr_loss=0.8666, tr_f1=0.6826, va_loss=2.5894, va_f1=0.4844
2026-07-08 10:30:57,774 - INFO - ANFIS Epoch 04: tr_loss=0.8165, tr_f1=0.7298, va_loss=1.3282, va_f1=0.1165
2026-07-08 10:30:58,173 - INFO - ANFIS Epoch 05: tr_loss=0.7975, tr_f1=0.7395, va_loss=22.8079, va_f1=0.0570
2026-07-08 10:30:58,581 - INFO - ANFIS Epoch 06: tr_loss=0.7907, tr_f1=0.7698, va_loss=8.1521, va_f1=0.4844
2026-07-08 10:30:58,983 - INFO - ANFIS Epoch 07: tr_loss=0.7332, tr_f1=0.7816, va_loss=8.1611, va_f1=0.0570
2026-07-08 10:30:59,374 - INFO - ANFIS Epoch 08: tr_loss=0.7349, tr_f1=0.7934, va_loss=0.5628, va_f1=0.7892
2026-07-08 10:30:59,787 - INFO - ANFIS Epoch 09: tr_loss=0.7053, tr_f1=0.7942, va_loss=4.0224, va_f1=0.0570
2026-07-08 10:31:00,186 - I

{'best_epoch': 61,
 'best_val_f1': 0.8520260434163587,
 'history': [{'epoch': 1,
   'train_loss': 1.0421862110495568,
   'train_acc': 0.4653,
   'train_f1_macro': 0.43212539597358746,
   'val_loss': 0.918846532702446,
   'val_acc': 0.9445,
   'val_f1_macro': 0.7490887513774689},
  {'epoch': 2,
   'train_loss': 0.9374258950352669,
   'train_acc': 0.7046,
   'train_f1_macro': 0.6080081207341358,
   'val_loss': 0.7746529430150986,
   'val_acc': 0.9395,
   'val_f1_macro': 0.48440319670018045},
  {'epoch': 3,
   'train_loss': 0.8666451901197434,
   'train_acc': 0.7896,
   'train_f1_macro': 0.6826084708178703,
   'val_loss': 2.5894424468278885,
   'val_acc': 0.9395,
   'val_f1_macro': 0.48440319670018045},
  {'epoch': 4,
   'train_loss': 0.8164630591869354,
   'train_acc': 0.836,
   'train_f1_macro': 0.7297613496113672,
   'val_loss': 1.3282309621572495,
   'val_acc': 0.1165,
   'val_f1_macro': 0.11649977912494477},
  {'epoch': 5,
   'train_loss': 0.7974607452750206,
   'train_acc': 0.8457,


# Tabla Comparativa de Baselines (Test)

In [10]:
import json
import os
import pandas as pd

model_dir = os.path.join("..", "models", "artifacts", "baseline_models")
metrics_dir = os.path.join("..", "models", "metrics")

baselines = {
    "Random Forest": "baseline_rf_metrics.json",
    "XGBoost": "baseline_xgb_metrics.json",
    "LSTM": "baseline_lstm_metrics.json",
    "ANFIS": "baseline_anfis_metrics.json",
}

rows = []
for name, fname in baselines.items():
    path = os.path.join(model_dir, fname)
    with open(path, "r") as f:
        data = json.load(f)
    test = data.get("test", {})
    rows.append({
        "Modelo": name,
        "Accuracy": round(test.get("accuracy", 0), 4),
        "AUC-ROC": round(test.get("auc_roc", test.get("roc_auc", 0)), 4),
        "F1 Fallo": round(test.get("fallo_f1", test.get("f1_binary", 0)), 4),
        "F1 Macro": round(test.get("macro_f1", test.get("f1_macro", 0)), 4),
        "Precision Fallo": round(test.get("fallo_precision", test.get("precision", 0)), 4),
        "Precision Macro": round(test.get("macro_precision", 0), 4),
        "Recall Fallo": round(test.get("fallo_recall", test.get("recall", 0)), 4),
        "Recall Macro": round(test.get("macro_recall", test.get("recall_macro", 0)), 4),
    })

# Add DNF row from results.json
dnf_path = os.path.join(metrics_dir, "results.json")
if os.path.exists(dnf_path):
    with open(dnf_path, "r") as f:
        dnf_data = json.load(f)
    dnf_test = dnf_data.get("test_metrics", {})
    rows.append({
        "Modelo": "DNF (seleccionado)",
        "Accuracy": round(dnf_test.get("accuracy", 0), 4),
        "AUC-ROC": round(dnf_test.get("fallo_auc", dnf_test.get("roc_auc", 0)), 4),
        "F1 Fallo": round(dnf_test.get("fallo_f1", 0), 4),
        "F1 Macro": round(dnf_test.get("macro_f1", 0), 4),
        "Precision Fallo": round(dnf_test.get("fallo_precision", 0), 4),
        "Precision Macro": round(dnf_test.get("macro_precision", 0), 4),
        "Recall Fallo": round(dnf_test.get("fallo_recall", 0), 4),
        "Recall Macro": round(dnf_test.get("macro_recall", 0), 4),
    })

df = pd.DataFrame(rows)
df = df.set_index("Modelo")
df.style.format("{:.4f}").highlight_null("red")


,Accuracy,AUC-ROC,F1 Fallo,F1 Macro,Precision Fallo,Precision Macro,Recall Fallo,Recall Macro
Modelo,,,,,,,,
Random Forest,0.9612,0.8372,0.4201,0.7000,0.9272,0.9445,0.2716,0.6352
XGBoost,0.9645,0.8504,0.4861,0.7339,0.9692,0.9668,0.3244,0.6619
LSTM,0.9720,0.9424,0.7375,0.8614,0.7919,0.8867,0.6901,0.8396
ANFIS,0.9563,0.9341,0.6700,0.8233,0.5885,0.7874,0.7778,0.8725
DNF (seleccionado),0.9890,0.9954,0.9054,0.9498,0.8876,0.9415,0.9240,0.9585
